# =============================================================================
# Example: open-system Heisenberg chain with single-qubit amplitude damping
# =============================================================================


METHODOLOGY NOTE: the dissipator gates built here (via
open_product_formula_generation.jl) use the DIRECT/brute-force route --
dense exponentiation of the local vectorized-Lindbladian generator, no
Kraus operators. This is the baseline to characterize before comparing
against a Kraus-operator-based construction (see kraus_channel_gate in
liouville_space.jl, not yet wired into the product formula).

In [1]:
using ITensors, ITensorMPS
using LinearAlgebra

include("F_diagnostics.jl")   # pulls in everything else via its include chain

F_report (generic function with 1 method)

In [9]:
import Distributions
import Random
Random.seed!(1234)
n = 3                     # number of system qubits
J = rand(Distributions.Uniform(1/4, 3/4), n-1)
gammas = fill(0.05, n)     # uniform amplitude-damping rate on every qubit
t = 3.0                    # total evolution time
ks = [3, 8]                # Trotter step counts to compare (k_i, k_j)
initially_excited = ["0", "2"]   # 0-indexed qubit labels that start in |1>


cutoff = 1e-10
maxdim = 200

lsites = liouville_siteinds(n)

LiouvilleSites(3, Index[(dim=2|id=362|"Ket,Qubit,n=1"), (dim=2|id=894|"Ket,Qubit,n=2"), (dim=2|id=242|"Ket,Qubit,n=3")], Index[(dim=2|id=425|"Bra,Qubit,n=1"), (dim=2|id=920|"Bra,Qubit,n=2"), (dim=2|id=317|"Bra,Qubit,n=3")])

In [10]:
# --- Build the full set of F_ij (Eq. 50) for all pairs in ks, via middle-out ---
Fs = build_open_F(n, J, gammas, t, ks, lsites, cutoff, maxdim; order=1)
println("Built ", length(Fs), " F_ij components for ks = ", ks)
for (idx, F) in enumerate(Fs)
    println("  F[$idx]: middle bond dim = ", middle_bond_dim(F))
end

# --- Build F_ex,j (Eq. 51) using a fine reference k0 >> ks ---
k0 = 10
Fex = build_open_F_ex(n, J, gammas, t, ks, k0, lsites, cutoff, maxdim; order=1, order_ref=2)
println("\nBuilt ", length(Fex), " F_ex,j components against reference k0 = ", k0)
for (idx, F) in enumerate(Fex)
    println("  F_ex[$idx]: middle bond dim = ", middle_bond_dim(F))
end

Built 4 F_ij components for ks = [3, 8]
  F[1]: middle bond dim = 16
  F[2]: middle bond dim = 16
  F[3]: middle bond dim = 16
  F[4]: middle bond dim = 16

Built 2 F_ex,j components against reference k0 = 10
  F_ex[1]: middle bond dim = 16
  F_ex[2]: middle bond dim = 16


In [11]:
# --- Track Schmidt-rank growth of F_ii layer by layer (the diagnostic the
#     project notes recommend running first, since the closed-system
#     near-identity argument is not expected to survive once gamma > 0) ---
k_track = 8
result = track_Fii_bond_dimension(
    n, J, gammas, t, k_track, lsites, cutoff, maxdim;
    order=1, track_exact_rank=true, rank_cutoff=1e-10
)

println("\nLayer-by-layer F_ii growth (n=$n, gamma=$(gammas[1]), t=$t, k=$k_track):")
println("layer | stored bond dim | exact operator-Schmidt rank")
for i in eachindex(result.layer)
    println(result.layer[i], "     | ", result.bond_dim[i], "       | ", result.schmidt_rank[i])
end

# Full bond-dimension profile of the final F_ii (all bonds, not just middle)
println("\nFull bond profile of final F_ii: ", full_bond_dimension_profile(result.F_final))



Layer-by-layer F_ii growth (n=3, gamma=0.05, t=3.0, k=8):
layer | stored bond dim | exact operator-Schmidt rank
1     | 16       | 16
2     | 16       | 16
3     | 16       | 16
4     | 16       | 16
5     | 16       | 16
6     | 16       | 16
7     | 16       | 16
8     | 16       | 16

Full bond profile of final F_ii: [16, 16]


In [12]:
# =============================================================================
# F(t) report: <<rho(0)|F_ij|rho(0)>> matrix and distance-to-identity,
# mirroring Eqs. 14-16 of main.pdf
# =============================================================================
#
# OPEN system (dissipation on): expect the diagonal entries to be visibly
# below 1 (and ||F_ii - Id||_F visibly above 0), unlike the closed-system
# case where F_ii is exactly the identity by construction.
 
report_open = F_report(
    n, J, gammas, t, ks, initially_excited, lsites, cutoff, maxdim;
    order=1, dissipation=true
)
 
println("\n--- Open system (dissipation ON, gamma = $(gammas[1])) ---")
println("<<rho(0)|F_ij|rho(0)>> matrix:")
display(report_open.M)
println("\n||F_ij - Id||_F matrix:")
display(report_open.distances)
 
# CLOSED system check (dissipation off): set dissipation=false to recover
# the ideal unitary Heisenberg evolution with NO code changes elsewhere --
# expect the diagonal of M to be numerically 1 and the diagonal of
# `distances` to be numerically 0, exactly as in the closed-system paper's
# Eq. 14 (e.g. F11 = F22 = 0.999999 there).
 
report_closed = F_report(
    n, J, gammas, t, ks, initially_excited, lsites, cutoff, maxdim;
    order=1, dissipation=false
)
 
println("\n--- Closed-system check (dissipation OFF) ---")
println("<<rho(0)|F_ij|rho(0)>> matrix:")
display(report_closed.M)
println("\n||F_ij - Id||_F matrix:")
display(report_closed.distances)

2×2 Matrix{ComplexF64}:
 0.570296+4.88286e-16im  0.535613-8.90983e-15im
 0.535613-2.77768e-15im  0.569789-1.30051e-14im

2×2 Matrix{Float64}:
 3.02031  3.66146
 3.66146  3.0181


--- Open system (dissipation ON, gamma = 0.05) ---
<<rho(0)|F_ij|rho(0)>> matrix:

||F_ij - Id||_F matrix:

--- Closed-system check (dissipation OFF) ---
<<rho(0)|F_ij|rho(0)>> matrix:

||F_ij - Id||_F matrix:


2×2 Matrix{ComplexF64}:
      1.0+3.95556e-17im  0.938085-1.30106e-14im
 0.938085-1.217e-15im         1.0+1.22613e-16im

2×2 Matrix{Float64}:
 5.0629e-14  2.57475
 2.57475     4.5356e-14